### Lead Scoring — PySpark (Databricks) Notebook

**Rule-based, two-layer lead scoring on top of the vulnerability CSVs.**

Upstream, the raw scan data (`test_scans.json`) is filtered/attributed to produce:
- `vulnerable_hosts.csv` — 1 row per vulnerable (host, port)
- `vulnerable_cves.csv`  — 1 row per (host, port, CVE)

This notebook doesn't drop any more records — it scores every host and
company from those two CSVs, then demotes (via a quality penalty) leads
that look like hosting providers or have weak company attribution, instead
of removing them outright.

It reads those CSVs from `data/filtered_data/` and produces two ranked outputs in the **same folder**:
- `hosts_scored.csv`  — Layer 1: per-host exposure score (0–100)
- `company_leads.csv` — Layer 2: per-company lead score, grade, and priority reason

Everything runs on Spark DataFrames — no per-row Python loops.

### 1. Imports, config & Spark session

In [1]:
import os

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType, IntegerType

try:
    spark  # provided automatically in Databricks / Jupyter-Spark
except NameError:
    spark = (
        SparkSession.builder
        .appName("lead_score")
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", "64")
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("WARN")

try:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
except Exception:
    PROJECT_ROOT = "/Users/vaddegurudevaraju/Desktop/CyberSecurity"

# OUTPUT_ROOT = this repo's root, one level above this notebook (pyspark_files/..).
# vulnerable_hosts.csv / vulnerable_cves.csv (read) and hosts_scored.csv /
# company_leads.csv (written) all live under OUTPUT_ROOT/data/, not the old repo.
try:
    OUTPUT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
except Exception:
    OUTPUT_ROOT = PROJECT_ROOT

INPUT_JSON  = os.path.join(PROJECT_ROOT, "data", "raw_data", "test_scans.json")
OUTPUT_DIR  = os.path.join(OUTPUT_ROOT, "data", "filtered_data")
HOSTS_CSV   = os.path.join(OUTPUT_DIR, "vulnerable_hosts.csv")
CVES_CSV    = os.path.join(OUTPUT_DIR, "vulnerable_cves.csv")
HOSTS_OUT   = os.path.join(OUTPUT_DIR, "hosts_scored.csv")
LEADS_OUT   = os.path.join(OUTPUT_DIR, "company_leads.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"OUTPUT_ROOT  = {OUTPUT_ROOT}")
print(f"INPUT_JSON   = {INPUT_JSON}")
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")
print(f"HOSTS_CSV    = {HOSTS_CSV}  (exists = {os.path.exists(HOSTS_CSV)})")
print(f"CVES_CSV     = {CVES_CSV}   (exists = {os.path.exists(CVES_CSV)})")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/14 16:26:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PROJECT_ROOT = /Users/vaddegurudevaraju/Desktop/CyberSecurity
INPUT_JSON   = /Users/vaddegurudevaraju/Desktop/CyberSecurity/data/test_scan.json
OUTPUT_DIR   = /Users/vaddegurudevaraju/Desktop/CyberSecurity/data/vulns_output
HOSTS_CSV    = /Users/vaddegurudevaraju/Desktop/CyberSecurity/data/vulns_output/vulnerable_hosts.csv  (exists = True)
CVES_CSV     = /Users/vaddegurudevaraju/Desktop/CyberSecurity/data/vulns_output/vulnerable_cves.csv   (exists = True)


### 2. Scoring weights & constants

Kept identical to the original `lead_score.py` so results are directly comparable.

In [2]:
W_BREADTH      = 0.30
W_SEVERITY     = 0.25
W_EXPLOIT      = 0.20
W_CONFIRMATION = 0.15
W_DEPTH        = 0.10

QUALITY_PENALTY = 0.30
CONF_THRESHOLD  = 0.40

print("Weights sum =", W_BREADTH + W_SEVERITY + W_EXPLOIT + W_CONFIRMATION + W_DEPTH)

Weights sum = 1.0


### 3. Load `vulnerable_hosts.csv` and `vulnerable_cves.csv`

In [3]:
hosts_df = (
    spark.read
         .option("header", True)
         .option("multiLine", True)
         .option("escape", '"')
         .csv(HOSTS_CSV)
)

cves_df = (
    spark.read
         .option("header", True)
         .option("multiLine", True)
         .option("escape", '"')
         .csv(CVES_CSV)
)

print(f"vulnerable_hosts rows = {hosts_df.count():,}")
print(f"vulnerable_cves  rows = {cves_df.count():,}")

hosts_df.printSchema()

vulnerable_hosts rows = 205,278


vulnerable_cves  rows = 7,576,007
root
 |-- ip_str: string (nullable = true)
 |-- port: string (nullable = true)
 |-- transport: string (nullable = true)
 |-- product: string (nullable = true)
 |-- version: string (nullable = true)
 |-- os: string (nullable = true)
 |-- org: string (nullable = true)
 |-- isp: string (nullable = true)
 |-- asn: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- _company: string (nullable = true)
 |-- _company_domain: string (nullable = true)
 |-- _company_source: string (nullable = true)
 |-- _company_confidence: string (nullable = true)
 |-- _company_is_provider: string (nullable = true)
 |-- _vuln_bucket: string (nullable = true)
 |-- _vuln_count: string (nullable = true)
 |-- _vuln_max_cvss: string (nullable = true)
 |-- _vuln_severity: string (nullable = true)
 |-- _vuln_cves: string (nullable = true)
 |-- _vuln_verified_cves: string (nullable = true)



### 4. Per-host max EPSS (from the CVE table)

For each `(ip_str, port)` we take the **maximum EPSS** across all its CVEs —
this is the `_host_max_epss` used in the Layer-1 host score.

In [4]:
host_epss_df = (
    cves_df
    .withColumn("epss_num", F.coalesce(F.col("epss").cast(DoubleType()), F.lit(0.0)))
    .groupBy("ip_str", "port")
    .agg(F.max("epss_num").alias("_host_max_epss"))
)

host_epss_df.show(5, truncate=False)

+-------------+----+--------------+
|ip_str       |port|_host_max_epss|
+-------------+----+--------------+
|3.169.173.46 |443 |0.94374       |
|54.252.22.37 |3790|0.01859       |
|66.70.154.157|80  |0.9446        |
|67.199.179.31|8172|0.94432       |
|143.14.210.24|3128|0.38209       |
+-------------+----+--------------+
only showing top 5 rows


### 5. Layer 1 — per-host exposure score (0–100)

```
severity = (cvss / 10) * 40      # 0-40
exploit  = max_epss   * 30       # 0-30
verified = 15 if any verified else 0
volume   = min(cve_count, 10)/10 * 15   # 0-15
host_score = clamp(severity + exploit + verified + volume, 0, 100)
```

In [5]:
cvss_col       = F.coalesce(F.col("_vuln_max_cvss").cast(DoubleType()), F.lit(0.0))
vuln_count_col = F.coalesce(F.col("_vuln_count").cast(IntegerType()),  F.lit(0))

cves_list_size     = F.size(F.expr("filter(split(coalesce(_vuln_cves, ''),          ';'), x -> x <> '')"))
verified_list_size = F.size(F.expr("filter(split(coalesce(_vuln_verified_cves, ''), ';'), x -> x <> '')"))

cve_count_eff = F.when(vuln_count_col > 0, vuln_count_col).otherwise(cves_list_size)
has_verified  = verified_list_size > 0

hosts_joined = hosts_df.join(host_epss_df, on=["ip_str", "port"], how="left") \
                        .withColumn("_host_max_epss", F.coalesce(F.col("_host_max_epss"), F.lit(0.0)))

severity_pts = (cvss_col / F.lit(10.0)) * F.lit(40.0)
exploit_pts  = F.col("_host_max_epss") * F.lit(30.0)
verified_pts = F.when(has_verified, F.lit(15.0)).otherwise(F.lit(0.0))
volume_pts   = (F.least(cve_count_eff, F.lit(10)).cast(DoubleType()) / F.lit(10.0)) * F.lit(15.0)

raw_score = severity_pts + exploit_pts + verified_pts + volume_pts
host_score = F.round(F.greatest(F.lit(0.0), F.least(F.lit(100.0), raw_score)), 1)

hosts_scored_df = hosts_joined.withColumn("_host_score", host_score)

hosts_scored_df.select(
    "ip_str", "port", "_company", "_vuln_max_cvss", "_vuln_count",
    "_host_max_epss", "_host_score"
).show(10, truncate=False)

+--------------+-----+-----------------------------+--------------+-----------+--------------+-----------+
|ip_str        |port |_company                     |_vuln_max_cvss|_vuln_count|_host_max_epss|_host_score|
+--------------+-----+-----------------------------+--------------+-----------+--------------+-----------+
|3.169.173.46  |443  |Kose                         |9.8           |28         |0.94374       |82.5       |
|54.252.22.37  |3790 |Jsdelivr                     |9.9           |9          |0.01859       |53.7       |
|66.70.154.157 |80   |Ip 66 70 154                 |7.7           |4          |0.9446        |65.1       |
|67.199.179.31 |8172 |Spanish Fork City            |9.8           |96         |0.94432       |82.5       |
|143.14.210.24 |3128 |Brander Group Inc            |10.0          |21         |0.38209       |66.5       |
|62.217.10.147 |9000 |GTT Communications France SAS|9.1           |3          |0.1294        |44.8       |
|38.154.30.26  |3128 |B2 Net Solution

### 6. Write `hosts_scored.csv`

We coalesce to a single partition and rename to a plain filename so the output matches the original script (`hosts_scored.csv` in `data/filtered_data/`).

In [6]:
import glob, shutil

def _write_single_csv(df, final_path):
    tmp_dir = final_path + "__tmp"
    (
        df.coalesce(1)
          .write.mode("overwrite")
          .option("header", True)
          .option("escape", '"')
          .csv(tmp_dir)
    )
    part = glob.glob(os.path.join(tmp_dir, "part-*.csv"))[0]
    if os.path.exists(final_path):
        os.remove(final_path)
    shutil.move(part, final_path)
    shutil.rmtree(tmp_dir, ignore_errors=True)

_write_single_csv(hosts_scored_df, HOSTS_OUT)
print(f"Wrote {HOSTS_OUT}")

Wrote /Users/vaddegurudevaraju/Desktop/CyberSecurity/data/vulns_output/hosts_scored.csv


### 7. Layer 2 — per-company aggregation

Group hosts by their inferred **company** (`_company_domain` if present, else `_company`, else `UNKNOWN`) and roll up the raw features.

In [7]:
company_key = F.lower(F.trim(
    F.coalesce(F.col("_company_domain"), F.col("_company"), F.lit("UNKNOWN"))
))

conf_col       = F.coalesce(F.col("_company_confidence").cast(DoubleType()), F.lit(0.0))
is_provider_c  = F.lower(F.coalesce(F.col("_company_is_provider").cast("string"), F.lit(""))) == F.lit("true")

hosts_for_agg = (
    hosts_scored_df
    .withColumn("_company_key",        company_key)
    .withColumn("_cve_count_eff",      cve_count_eff)
    .withColumn("_verified_count_eff", verified_list_size)
    .withColumn("_cvss_num",           cvss_col)
    .withColumn("_conf_num",           conf_col)
    .withColumn("_is_provider_flag",   is_provider_c.cast(IntegerType()))
    .withColumn("_is_critical",        (cvss_col >= F.lit(9.0)).cast(IntegerType()))
    .withColumn("_is_high",            ((cvss_col >= F.lit(7.0)) & (cvss_col <  F.lit(9.0))).cast(IntegerType()))
)

company_agg = (
    hosts_for_agg.groupBy("_company_key")
    .agg(
        F.countDistinct(F.struct("ip_str", "port")).alias("n_hosts"),
        F.sum("_cve_count_eff").alias("n_cves"),
        F.sum("_verified_count_eff").alias("n_verified_cves"),
        F.sum("_is_critical").alias("n_critical"),
        F.sum("_is_high").alias("n_high"),
        F.max("_cvss_num").alias("max_cvss"),
        F.max("_host_max_epss").alias("max_epss"),
        F.avg("_conf_num").alias("avg_confidence"),
        F.sum("_is_provider_flag").alias("_provider_votes"),
        F.count(F.lit(1)).alias("_n_rows"),
        F.first("_company",        ignorenulls=True).alias("company"),
        F.first("_company_domain", ignorenulls=True).alias("domain"),
    )
)

company_agg.show(5, truncate=False)

+------------------------------------------------------------------------------------------------+-------+------+---------------+----------+------+--------+--------+--------------+---------------+-------+------------------------------------------------------------------------------------------------+------+
|_company_key                                                                                    |n_hosts|n_cves|n_verified_cves|n_critical|n_high|max_cvss|max_epss|avg_confidence|_provider_votes|_n_rows|company                                                                                         |domain|
+------------------------------------------------------------------------------------------------+-------+------+---------------+----------+------+--------+--------+--------------+---------------+-------+------------------------------------------------------------------------------------------------+------+
|&#24191;&#19996;&#30408;&#20449;&#20449;&#24687;&#25237;&#36164;&#26377;

### 8. Layer 2 — lead score, grade & priority reason

```
breadth      = min(n_hosts, 20) / 20
severity     = max_cvss / 10
exploit      = max_epss
confirmation = min(n_verified, 5) / 5
depth        = min(n_cves, 50) / 50

lead_score = 100 * (0.30*breadth + 0.25*severity + 0.20*exploit
                  + 0.15*confirmation + 0.10*depth)

if is_provider or avg_confidence < 0.40:
    lead_score *= 0.30   # quality penalty
```

In [8]:
breadth      = F.least(F.col("n_hosts"),         F.lit(20)).cast(DoubleType()) / F.lit(20.0)
severity     = F.coalesce(F.col("max_cvss"),    F.lit(0.0)) / F.lit(10.0)
exploit      = F.coalesce(F.col("max_epss"),    F.lit(0.0))
confirmation = F.least(F.col("n_verified_cves"), F.lit(5)).cast(DoubleType()) / F.lit(5.0)
depth        = F.least(F.col("n_cves"),          F.lit(50)).cast(DoubleType()) / F.lit(50.0)

raw_lead = F.lit(100.0) * (
    F.lit(W_BREADTH)      * breadth      +
    F.lit(W_SEVERITY)     * severity     +
    F.lit(W_EXPLOIT)      * exploit      +
    F.lit(W_CONFIRMATION) * confirmation +
    F.lit(W_DEPTH)        * depth
)

is_provider  = F.col("_provider_votes") > (F.col("_n_rows") / F.lit(2.0))
low_conf     = F.col("avg_confidence") < F.lit(CONF_THRESHOLD)
penalized    = is_provider | low_conf

penalized_score = F.when(penalized, raw_lead * F.lit(QUALITY_PENALTY)).otherwise(raw_lead)
lead_score      = F.round(F.greatest(F.lit(0.0), F.least(F.lit(100.0), penalized_score)), 1)

lead_grade = (
    F.when(lead_score >= 75, F.lit("A"))
     .when(lead_score >= 50, F.lit("B"))
     .when(lead_score >= 25, F.lit("C"))
     .otherwise(F.lit("D"))
)

reason_parts = F.array(
    F.when(F.col("n_critical") > 0, F.concat(F.col("n_critical").cast("string"), F.lit(" critical CVE host(s)"))),
    F.when(F.col("n_high")     > 0, F.concat(F.col("n_high").cast("string"),     F.lit(" high-severity host(s)"))),
    F.concat(F.col("n_hosts").cast("string"), F.lit(" exposed host(s)")),
    F.when(F.col("max_epss") >= 0.5,
           F.concat(F.lit("high exploit likelihood (EPSS "), F.format_number(F.col("max_epss"), 2), F.lit(")"))),
    F.when(F.col("n_verified_cves") > 0,
           F.concat(F.col("n_verified_cves").cast("string"), F.lit(" verified CVE(s)"))),
    F.when(penalized, F.lit("LOW-QUALITY lead (hosting provider / weak attribution)")),
)

priority_reason = F.concat_ws("; ", F.expr("filter(reason_parts_arr, x -> x is not null)"))

leads_df = (
    company_agg
    .withColumn("reason_parts_arr", reason_parts)
    .withColumn("is_provider",      is_provider)
    .withColumn("lead_score",       lead_score)
    .withColumn("lead_grade",       lead_grade)
    .withColumn("priority_reason",  priority_reason)
    .withColumn("max_cvss",         F.round(F.coalesce(F.col("max_cvss"), F.lit(0.0)), 1))
    .withColumn("max_epss",         F.round(F.coalesce(F.col("max_epss"), F.lit(0.0)), 3))
    .withColumn("avg_confidence",   F.round(F.coalesce(F.col("avg_confidence"), F.lit(0.0)), 2))
    .withColumn("company",          F.coalesce(F.col("company"), F.col("_company_key")))
    .withColumn("domain",           F.coalesce(F.col("domain"),  F.lit("")))
    .select(
        "company", "domain", "n_hosts", "n_cves", "n_verified_cves",
        "n_critical", "n_high", "max_cvss", "max_epss", "avg_confidence",
        "is_provider", "lead_score", "lead_grade", "priority_reason",
    )
    .orderBy(F.col("lead_score").desc())
)

leads_df.show(10, truncate=False)

+-------------------------------------------------------+---------------------+-------+------+---------------+----------+------+--------+--------+--------------+-----------+----------+----------+-------------------------------------------------------------------------------------------------------------------------------------+
|company                                                |domain               |n_hosts|n_cves|n_verified_cves|n_critical|n_high|max_cvss|max_epss|avg_confidence|is_provider|lead_score|lead_grade|priority_reason                                                                                                                      |
+-------------------------------------------------------+---------------------+-------+------+---------------+----------+------+--------+--------+--------------+-----------+----------+----------+-------------------------------------------------------------------------------------------------------------------------------------+
|Oracle Co

### 9. Write `company_leads.csv`

In [9]:
_write_single_csv(leads_df, LEADS_OUT)
print(f"Wrote {LEADS_OUT}")

Wrote /Users/vaddegurudevaraju/Desktop/CyberSecurity/data/vulns_output/company_leads.csv


### 10. Run summary

In [10]:
n_hosts_scored = hosts_scored_df.count()
n_companies    = leads_df.count()

grade_dist = (
    leads_df.groupBy("lead_grade")
            .agg(F.count(F.lit(1)).alias("n"))
            .orderBy("lead_grade")
)

print(f"Scored {n_hosts_scored:,} hosts   -> {HOSTS_OUT}")
print(f"Ranked {n_companies:,} companies  -> {LEADS_OUT}")
print("\nLead grade distribution:")
grade_dist.show(truncate=False)

print("Top 10 leads:")
leads_df.select("lead_grade", "lead_score", "company", "domain", "n_hosts", "n_cves", "priority_reason") \
        .show(10, truncate=False)

Scored 205,278 hosts   -> /Users/vaddegurudevaraju/Desktop/CyberSecurity/data/vulns_output/hosts_scored.csv
Ranked 49,161 companies  -> /Users/vaddegurudevaraju/Desktop/CyberSecurity/data/vulns_output/company_leads.csv

Lead grade distribution:


+----------+-----+
|lead_grade|n    |
+----------+-----+
|A         |646  |
|B         |16003|
|C         |28293|
|D         |4219 |
+----------+-----+

Top 10 leads:


+----------+----------+-------------------------------------------------------+---------------------+-------+------+-------------------------------------------------------------------------------------------------------------------------------------+
|lead_grade|lead_score|company                                                |domain               |n_hosts|n_cves|priority_reason                                                                                                                      |
+----------+----------+-------------------------------------------------------+---------------------+-------+------+-------------------------------------------------------------------------------------------------------------------------------------+
|A         |98.9      |Oracle Corporation                                     |                     |773    |12979 |444 critical CVE host(s); 343 high-severity host(s); 773 exposed host(s); high exploit likelihood (EPSS 0.94); 6 verified CVE(s)   